# Pertemuan 5 — Tugas 2: DBSCAN pada Make Moons

Tugas ini menggunakan 1.000 sampel berbentuk dua bulan sabit dengan noise 0.05. Kita menjalankan model awal, menghitung enam metrik, membuat visualisasi, lalu membandingkan parameter.

Sumber: [Tugas Praktikum](https://polinema.gitbook.io/jti-modul-praktikum-pembelajaran-mesin-2025-2026/js04-klasterisasi/tugas-praktikum). Dataset dibuat dengan `make_moons`, sehingga tidak memerlukan file unduhan.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn import metrics


## 1. Membuat dan menormalisasi data
Kita memakai StandardScaler seperti contoh DBSCAN pada modul. Label asli disimpan untuk evaluasi dan tidak diberikan ke model.


In [2]:
X_raw, labels_true = make_moons(n_samples=1000, noise=0.05, random_state=0)
X = StandardScaler().fit_transform(X_raw)
plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], s=15)
plt.xlabel('Fitur 1 (standar)')
plt.ylabel('Fitur 2 (standar)')
plt.title('Make Moons: 1.000 Sampel')
plt.show()


## 2. Menyiapkan fungsi evaluasi
Jumlah klaster tidak memasukkan label noise `-1`. Homogeneity, Completeness, V-measure, ARI, dan AMI dihitung pada semua titik agar sampel noise tetap memengaruhi penilaian.

Silhouette utama dihitung tanpa noise karena noise bukan satu klaster biasa. Sebagai pembanding dengan contoh modul, tabel juga memuat Silhouette semua titik. Jika jumlah klaster tidak memenuhi syarat, nilainya `NaN` (tidak dapat dihitung), bukan nol. Silhouette tidak selalu cocok menjadi satu-satunya patokan untuk kelompok yang melengkung.


In [3]:
def evaluasi_dbscan(X, labels_true, eps, min_samples):
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    labels = db.labels_
    mask = labels != -1
    jumlah_klaster = len(set(labels[mask]))
    sil = np.nan
    if 2 <= jumlah_klaster < int(mask.sum()):
        sil = metrics.silhouette_score(X[mask], labels[mask])
    sil_semua = np.nan
    if 2 <= len(set(labels)) < len(labels):
        sil_semua = metrics.silhouette_score(X, labels)
    hasil = {
        'eps': eps, 'min_samples': min_samples,
        'Klaster': jumlah_klaster, 'Noise': int(np.sum(~mask)),
        'Homogeneity': metrics.homogeneity_score(labels_true, labels),
        'Completeness': metrics.completeness_score(labels_true, labels),
        'V-measure': metrics.v_measure_score(labels_true, labels),
        'ARI': metrics.adjusted_rand_score(labels_true, labels),
        'AMI': metrics.adjusted_mutual_info_score(labels_true, labels),
        'Silhouette': sil, 'Silhouette_semua_titik': sil_semua,
    }
    return db, hasil


## 3. Model awal: eps = 0.2 dan min_samples = 5
Kita menampilkan jumlah klaster, noise, dan seluruh metrik sesuai ketentuan tugas.


In [4]:
db_awal, hasil_awal = evaluasi_dbscan(X, labels_true, eps=0.2, min_samples=5)
display(pd.DataFrame([hasil_awal]).round(4))
print(f"Jumlah klaster: {hasil_awal['Klaster']}; noise: {hasil_awal['Noise']}")


Jumlah klaster: 2; noise: 0


## 4. Visualisasi hasil awal
Core sample ditampilkan sebagai titik besar, non-core sebagai titik kecil, dan noise sebagai titik hitam.


In [5]:
def plot_dbscan(X, db, ax, title):
    labels = db.labels_
    core_mask = np.zeros_like(labels, dtype=bool)
    core_mask[db.core_sample_indices_] = True
    unique_labels = sorted(set(labels))
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    for label, color in zip(unique_labels, colors):
        if label == -1:
            color = 'black'
        member = labels == label
        core = member & core_mask
        border = member & ~core_mask
        ax.scatter(X[core, 0], X[core, 1], c=[color], s=70, edgecolors='k', linewidths=0.3)
        ax.scatter(X[border, 0], X[border, 1], c=[color], s=15, edgecolors='k', linewidths=0.3)
    ax.set_title(title)
    ax.set_xlabel('Fitur 1 (standar)')
    ax.set_ylabel('Fitur 2 (standar)')


In [6]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_dbscan(X, db_awal, ax, 'DBSCAN Awal: eps = 0.2, min_samples = 5')
plt.show()


## 5. Eksperimen parameter
Kita mencoba semua kombinasi `eps = 0.05, 0.1, 0.3, 0.5` dan `min_samples = 3, 10, 20` (12 kombinasi). Untuk melihat pengaruh setiap parameter secara terpisah, kita juga mencoba perubahan eps dengan min_samples tetap 5 dan perubahan min_samples dengan eps tetap 0.2.

Semua eksperimen memakai data yang sama agar hasilnya bisa dibandingkan.


In [7]:
eps_list = [0.05, 0.1, 0.3, 0.5]
min_list = [3, 10, 20]
kombinasi = [(0.2, 5)]
kombinasi += [(eps, minimum) for eps in eps_list for minimum in min_list]
kombinasi += [(eps, 5) for eps in eps_list]
kombinasi += [(0.2, minimum) for minimum in min_list]
hasil_eksperimen = []
model_eksperimen = {}
for eps, minimum in kombinasi:
    db, hasil = evaluasi_dbscan(X, labels_true, eps, minimum)
    hasil_eksperimen.append(hasil)
    model_eksperimen[(eps, minimum)] = db
tabel = pd.DataFrame(hasil_eksperimen)
display(tabel.round(4))
tabel.to_csv('hasil_eksperimen_dbscan.csv', index=False)


### Perbandingan visual 12 kombinasi
Ukuran titik dan warna noise memakai aturan yang sama dengan visualisasi model awal.


In [8]:
fig, axes = plt.subplots(4, 3, figsize=(15, 17))
for i, eps in enumerate(eps_list):
    for j, minimum in enumerate(min_list):
        db = model_eksperimen[(eps, minimum)]
        row = tabel[(tabel['eps'] == eps) & (tabel['min_samples'] == minimum)].iloc[0]
        title = (f'eps={eps}, min_samples={minimum}\n'
                 f"{int(row['Klaster'])} klaster, {int(row['Noise'])} noise")
        plot_dbscan(X, db, axes[i, j], title)
plt.tight_layout()
plt.show()


### Mengamati perubahan satu parameter
Tabel berikut memisahkan perubahan eps dan min_samples sehingga lebih mudah dijelaskan.


In [9]:
print('Perubahan eps, min_samples tetap 5:')
display(tabel[tabel['min_samples'] == 5].sort_values('eps').round(4))
print('Perubahan min_samples, eps tetap 0.2:')
display(tabel[tabel['eps'] == 0.2].sort_values('min_samples').round(4))

terbaik = tabel.sort_values(['ARI', 'Noise'], ascending=[False, True]).iloc[0]
print('Salah satu hasil dengan ARI tertinggi:')
print(f"eps={terbaik['eps']}, min_samples={int(terbaik['min_samples'])}, "
      f"klaster={int(terbaik['Klaster'])}, noise={int(terbaik['Noise'])}, "
      f"ARI={terbaik['ARI']:.4f}.")
print('ARI bisa digunakan di sini karena label asli data buatan tersedia.')


Perubahan eps, min_samples tetap 5:


Perubahan min_samples, eps tetap 0.2:


Salah satu hasil dengan ARI tertinggi:
eps=0.2, min_samples=5, klaster=2, noise=0, ARI=1.0000.
ARI bisa digunakan di sini karena label asli data buatan tersedia.


## Kesimpulan
Nilai eps kecil membuat radius tetangga sempit sehingga data dapat terpecah menjadi banyak kelompok atau noise. Nilai eps yang terlalu besar dapat menggabungkan dua kelompok. Jika min_samples dinaikkan pada eps yang sama, syarat kepadatan menjadi lebih ketat sehingga noise dapat bertambah.

Hasil yang sesuai untuk data ini sebaiknya memiliki dua klaster, sedikit noise, serta kecocokan tinggi dengan label asli. Jumlah klaster, noise, metrik, dan grafik perlu dilihat bersama. Rincian hasil tersimpan pada `hasil_eksperimen_dbscan.csv`.

Pada model awal (eps = 0.2, min_samples = 5), terbentuk 2 klaster tanpa noise dengan ARI = 1. Pada eps = 0.05 dan min_samples = 10, semua 1.000 titik menjadi noise. Pada eps = 0.5 dan min_samples = 3, kedua kelompok tergabung menjadi satu klaster. Contoh ini menunjukkan bahwa parameter yang terlalu kecil atau besar dapat mengubah hasil secara jelas.
